# 🤖 Linguistic Agent — Phase 2: BERT Fine-Tuning (TPU v2)

**Run this on Account B using a TPU v2 instance.**

## What We Are Doing (Explained)

### Method: Sequence Classification Fine-Tuning
We start with `bert-base-uncased`, a general-purpose language model pre-trained on
Wikipedia and BookCorpus. It already understands English grammar, context, and semantics.

We attach a **2-class linear classification head** on top and fine-tune the entire model
on our `transcripts.csv` dataset to teach it to distinguish bonafide vs spoof speech patterns.

### Why This Works
Synthesized (spoof) speech often has unnatural sentence structures, repetitive phrasing,
or semantically hollow content compared to natural human speech.

### Class Imbalance Strategy
We use `WeightedRandomSampler` on the training set. This oversamples the minority class
(bonafide) so the model sees balanced batches during training, even though the raw dataset
is ~1:9 bonafide:spoof.

### TPU Strategy
HuggingFace `Trainer` with `accelerate` automatically distributes computation across
all 8 TPU cores using PyTorch/XLA. Each core processes a sub-batch; gradients are
synchronized after each step.

### Failsafe
The model checkpoints every epoch to Google Drive. If the Colab instance dies,
just re-run from the training cell and set `resume_from_checkpoint=True`.

In [ ]:
!pip install -q transformers datasets evaluate accelerate scikit-learn scipy

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import torch
from pathlib import Path

DRIVE_DIR        = Path('/content/drive/MyDrive/40_PER_22_Data')
TRANSCRIPTS_CSV  = DRIVE_DIR / 'transcripts.csv'
MODEL_SAVE_DIR   = DRIVE_DIR / 'linguistic_bert_model'
CHECKPOINT_DIR   = DRIVE_DIR / 'linguistic_bert_checkpoints'
MODEL_SAVE_DIR.mkdir(exist_ok=True)
CHECKPOINT_DIR.mkdir(exist_ok=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'PyTorch device: {device}')

In [ ]:
import pandas as pd
from datasets import Dataset

df = pd.read_csv(TRANSCRIPTS_CSV)
df = df.dropna(subset=['text'])
df = df[df['text'].str.strip().astype(bool)].reset_index(drop=True)

train_df = df[df['split'] == 'train'].reset_index(drop=True)
test_df  = df[df['split'] == 'test'].reset_index(drop=True)

n_spoof    = int((train_df['label'] == 1).sum())
n_bonafide = int((train_df['label'] == 0).sum())
print(f'Train => bonafide: {n_bonafide:,}  |  spoof: {n_spoof:,}')
print(f'Test size: {len(test_df):,}')

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

def tokenize(examples):
    return tokenizer(
        examples['text'],
        padding='max_length',
        truncation=True,
        max_length=128,  # Transcripts are short — 128 tokens is more than enough
    )

KEEP_COLS = ['label']
def prep_dataset(df_in):
    ds = Dataset.from_pandas(df_in[['text', 'label']])
    ds = ds.map(tokenize, batched=True)
    ds = ds.rename_column('label', 'labels')
    ds = ds.remove_columns([c for c in ds.column_names if c not in ['input_ids', 'attention_mask', 'token_type_ids', 'labels']])
    ds.set_format('torch')
    return ds

train_dataset = prep_dataset(train_df)
test_dataset  = prep_dataset(test_df)
print(f'✅ Tokenization done. Train: {len(train_dataset):,} | Test: {len(test_dataset):,}')

In [ ]:
import torch
from torch.utils.data import WeightedRandomSampler

# Compute per-sample weights for oversampling minority class (bonafide)
labels      = train_df['label'].values
class_count = [n_bonafide, n_spoof]
weights     = [1.0 / class_count[l] for l in labels]
sampler     = WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)
print(f'✅ WeightedRandomSampler created. Bonafide weight: {1/n_bonafide:.6f} | Spoof weight: {1/n_spoof:.6f}')

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, roc_curve
from scipy.optimize import brentq
from scipy.interpolate import interp1d

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds  = np.argmax(logits, axis=-1)
    probs  = torch.softmax(torch.tensor(logits), dim=-1).numpy()[:, 1]

    acc = accuracy_score(labels, preds)
    f1  = f1_score(labels, preds, zero_division=0)
    try:
        auc = roc_auc_score(labels, probs)
        fpr, tpr, _ = roc_curve(labels, probs)
        fnr = 1 - tpr
        eer = brentq(lambda x: interp1d(fpr, fnr - fpr)(x), 0, 1)
    except Exception:
        auc = 0.5
        eer = 0.5

    return {'accuracy': acc, 'f1': f1, 'auc': auc, 'eer': eer}

In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

model = AutoModelForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

training_args = TrainingArguments(
    output_dir                  = str(CHECKPOINT_DIR),   # Checkpoints go to Drive
    num_train_epochs            = 4,
    per_device_train_batch_size = 32,                    # TPU loves large batches
    per_device_eval_batch_size  = 64,
    learning_rate               = 2e-5,
    weight_decay                = 0.01,
    warmup_ratio                = 0.1,
    evaluation_strategy         = 'epoch',
    save_strategy               = 'epoch',               # FAILSAFE: save every epoch
    load_best_model_at_end      = True,
    metric_for_best_model       = 'eer',
    greater_is_better           = False,                 # Lower EER = better
    fp16                        = device == 'cuda',      # Mixed precision on GPU only
    dataloader_num_workers      = 2,
    report_to                   = 'none',
    logging_steps               = 100,
)

class BalancedTrainer(Trainer):
    """Override get_train_dataloader to inject WeightedRandomSampler."""
    def get_train_dataloader(self):
        from torch.utils.data import DataLoader
        return DataLoader(
            self.train_dataset,
            batch_size = self.args.per_device_train_batch_size,
            sampler    = sampler,
            collate_fn = self.data_collator,
            drop_last  = self.args.dataloader_drop_last,
            num_workers= self.args.dataloader_num_workers,
        )

trainer = BalancedTrainer(
    model           = model,
    args            = training_args,
    train_dataset   = train_dataset,
    eval_dataset    = test_dataset,
    compute_metrics = compute_metrics,
)

print('Starting BERT fine-tuning...')
# Set resume_from_checkpoint=True to resume after a Colab crash
trainer.train(resume_from_checkpoint=False)

In [ ]:
import json

# FAILSAFE: Always save best model to Drive before disconnecting
print(f'Saving fine-tuned model to {MODEL_SAVE_DIR}...')
trainer.save_model(str(MODEL_SAVE_DIR))
tokenizer.save_pretrained(str(MODEL_SAVE_DIR))

eval_results = trainer.evaluate()
print('\n=== Final Test Results ===')
print(f"Accuracy : {eval_results['eval_accuracy']*100:.2f}%")
print(f"EER      : {eval_results['eval_eer']*100:.2f}%")
print(f"AUC      : {eval_results['eval_auc']:.4f}")
print(f"F1       : {eval_results['eval_f1']:.4f}")

with open(MODEL_SAVE_DIR / 'results.json', 'w') as f:
    json.dump(eval_results, f, indent=4)

print('✅ Done! Model and results safely saved to Google Drive.')